# 36 · Streaming inside the agent loop
Level L4 · Part 5 · professional · ~35 min · builds on 11, 35

## Goal
Make Scout's loop stream: print text as it arrives, collect tool-call arguments that arrive in
pieces, parse them only once the call is complete, and retry a tool call that `max_tokens` cut off
instead of running it.

## The idea
In lesson 11 Scout streamed plain text: join the chunks and you are done. Inside the agent loop the
reply can also hold tool calls, and their arguments stream too. Anthropic sends them as **partial
JSON**: pieces of text such as `{"filena` that are not valid JSON until the last piece arrives.
Gemini on Vertex sends them as values tagged with a path such as `$.text`. Either way you collect
the pieces per call, wait for the "call complete" signal, then parse. If the output limit runs out
first, the call is broken: you must not run it.

Analogy: a form being dictated over the phone. You can show it filling in, but you only post it
once the caller says "that's everything".

## Picture
```mermaid
flowchart LR
    S["stream events"] --> T{"event type"}
    T -- "text delta" --> P["print now"]
    T -- "tool call start" --> B["new buffer<br/>(id, name)"]
    T -- "args delta" --> A["append to buffer<br/>(preview only)"]
    T -- "stop reason" --> D{"max_tokens?"}
    D -- "no" --> J["parse buffers<br/>run tools"]
    D -- "yes, call open" --> R["discard, retry<br/>with bigger budget"]
```

## Already know this?
This is the first lesson of Part 5. If you can answer all of these, skip to lesson 45 (the checkpoint).
1. Tool arguments stream in as pieces. When may you parse them, and what do you do if `max_tokens`
   cut the call?
2. How would you make one agent loop run unchanged on two vendors and a fake model?
3. You run 50 conversations at once. What stops you hitting the rate limit, and how do you retry a
   tool that sends an email without sending it twice?
4. Why can thinking tokens make a run cost more than its visible output suggests?
5. Your prompt-cache hit rate is 0%. Name two likely causes. When is a batch API worth its delay?

## Step by step
### 1. Start the recorder
As in lesson 35, every Gemini call is recorded once to
`../data/llm_cassettes/36_streaming-tool-calls.yaml` and replayed after that, so no key is needed.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("36_streaming-tool-calls")

LLM mode: replay (36_streaming-tool-calls.yaml)


### 2. Setup from lesson 35 (shortened)
The notes and three tools. `search_notes` takes only `query` here (no pages), and the schemas are
written by hand instead of with the registry, to keep the setup short.

In [2]:
import ast, json, operator, os
from pathlib import Path

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
SCOUT_DIR = Path("../data/scout_files")
SCOUT_DIR.mkdir(parents=True, exist_ok=True)
for old in SCOUT_DIR.glob("*.txt"):
    old.unlink()

def search_notes(query: str) -> list[dict]:
    words = query.lower().split()
    return [n for n in NOTES if all(w in n["text"].lower() for w in words)]

print(search_notes("day on Mars"))

[{'id': 3, 'text': 'A day on Mars lasts about 24 hours and 39 minutes.'}]


The calculator walks the syntax tree with an allow-list (no `eval`), and `write_file` cannot leave
Scout's folder, as in lesson 35.

In [3]:
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def evaluate(node):
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in OPS:
        return OPS[type(node.op)](evaluate(node.left), evaluate(node.right))
    raise ValueError("use numbers, + - * / and brackets, e.g. '24*60+39'")

def calculator(expression: str) -> float:
    return evaluate(ast.parse(expression, mode="eval").body)

def write_file(filename: str, text: str, overwrite: bool = False) -> str:
    target = (SCOUT_DIR / filename).resolve()
    if target.parent != SCOUT_DIR.resolve():
        raise ValueError(f"'{filename}' is outside Scout's folder; use a plain file name")
    if target.exists() and not overwrite:
        raise FileExistsError(f"'{filename}' exists; pick a new name or set overwrite to true")
    target.write_text(text)
    return f"saved {len(text)} characters to {target.name}"

print(calculator("24*60+39"))

1479


The schemas and lesson 35's `run_tool`, which never raises.

In [4]:
def schema(name, description, **props):
    return {"name": name, "description": description, "parameters": {
        "type": "object", "properties": {k: {"type": t} for k, t in props.items()},
        "required": [k for k in props if k != "overwrite"]}}

SCHEMAS = [schema("search_notes", "Search the user's notes. Returns matching notes.", query="string"),
           schema("calculator", "Evaluate arithmetic like '24*60+39'.", expression="string"),
           schema("write_file", "Save text to a file in Scout's folder.",
                  filename="string", text="string", overwrite="boolean")]
TOOLS = {"search_notes": search_notes, "calculator": calculator, "write_file": write_file}

def run_tool(call: dict, tools: dict) -> dict:
    fn = tools.get(call["name"])
    if fn is None:
        return {"content": f"unknown tool '{call['name']}'. Available: {', '.join(tools)}", "is_error": True}
    try:
        return {"content": json.dumps(fn(**call["args"])), "is_error": False}
    except Exception as e:
        return {"content": f"{type(e).__name__}: {e}", "is_error": True}

print(run_tool({"name": "calculator", "args": {"expression": "1479/0"}}, TOOLS))

{'content': 'ZeroDivisionError: division by zero', 'is_error': True}


### 3. Stream event types
A streamed reply is a sequence of **events**, each with a `type`. We fake Anthropic's events
as shown in its streaming docs (a cut stream is our own assumption): `message_start`, then for each content block a
`content_block_start`, some `content_block_delta`s and a `content_block_stop`, then `message_delta`
with the stop reason, then `message_stop`. Text arrives as `text_delta`; tool arguments as
`input_json_delta` holding a `partial_json` string. Our fake counts each delta as one token.

In [5]:
def end(reason: str, used: int):
    yield {"type": "message_delta", "delta": {"stop_reason": reason}, "usage": {"output_tokens": used}}
    yield {"type": "message_stop"}

def fake_events(reply: dict, piece: int = 12, max_tokens: int = 10_000, turn: int = 1):
    """Stream a reply {"text", "tool_calls"} as Anthropic-style events, cut after max_tokens deltas."""
    blocks = ([("text", reply["text"])] if reply.get("text") else []) + [
        (c, json.dumps(c["args"])) for c in reply.get("tool_calls", [])]
    used = 0
    yield {"type": "message_start", "message": {"role": "assistant", "content": []}}
    for index, (kind, body) in enumerate(blocks):
        start = ({"type": "text", "text": ""} if kind == "text" else
                 {"type": "tool_use", "id": f"toolu_{turn}{index}", "name": kind["name"], "input": {}})
        yield {"type": "content_block_start", "index": index, "content_block": start}
        for i in range(0, len(body), piece):
            if used == max_tokens:  # cut stream: the docs do not say whether a stop event arrives
                yield from end("max_tokens", used); return
            used += 1
            delta = ({"type": "text_delta", "text": body[i:i + piece]} if kind == "text" else
                     {"type": "input_json_delta", "partial_json": body[i:i + piece]})
            yield {"type": "content_block_delta", "index": index, "delta": delta}
        yield {"type": "content_block_stop", "index": index}
    yield from end("tool_use" if reply.get("tool_calls") else "end_turn", used)

Scout says one short sentence, then asks to save a file. Print every event in one line each.

In [6]:
SAVE = {"text": "Saving it now.", "tool_calls": [{"name": "write_file", "args": {
    "filename": "mars_day.txt", "text": "A day on Mars lasts about 1479 minutes."}}]}
events = list(fake_events(SAVE))
for ev in events:
    detail = ev.get("delta") or ev.get("content_block") or ""
    print(f"{ev['type']:20} {ev.get('index', ''):1}  {json.dumps(detail)[:80]}")

message_start           ""
content_block_start  0  {"type": "text", "text": ""}
content_block_delta  0  {"type": "text_delta", "text": "Saving it no"}
content_block_delta  0  {"type": "text_delta", "text": "w."}
content_block_stop   0  ""
content_block_start  1  {"type": "tool_use", "id": "toolu_11", "name": "write_file", "input": {}}
content_block_delta  1  {"type": "input_json_delta", "partial_json": "{\"filename\":"}
content_block_delta  1  {"type": "input_json_delta", "partial_json": " \"mars_day.t"}
content_block_delta  1  {"type": "input_json_delta", "partial_json": "xt\", \"text\":"}
content_block_delta  1  {"type": "input_json_delta", "partial_json": " \"A day on M"}
content_block_delta  1  {"type": "input_json_delta", "partial_json": "ars lasts ab"}
content_block_delta  1  {"type": "input_json_delta", "partial_json": "out 1479 min"}
content_block_delta  1  {"type": "input_json_delta", "partial_json": "utes.\"}"}
content_block_stop   1  ""
message_delta           {"stop_reason"

Two content blocks: index 0 is text, index 1 is the tool call. Only `content_block_start` carries the
call's `id` and `name`; the deltas carry just the `index`. No single `partial_json` piece is valid
JSON: `"{\"filename\":"` is not an object yet.

### 4. Assemble the reply
Keep one buffer per tool block, keyed by **index**. Print text the moment it arrives. Append JSON
pieces to their buffer. Note when a block stops and what the stop reason is. Parse nothing yet.

In [7]:
def assemble(events, on_text=None, on_args=None) -> dict:
    on_text = on_text or (lambda t: print(t, end="", flush=True))
    text, calls, stop = "", {}, None
    for ev in events:
        if ev["type"] == "content_block_start" and ev["content_block"]["type"] == "tool_use":
            block = ev["content_block"]
            calls[ev["index"]] = {"id": block["id"], "name": block["name"], "json": "", "closed": False}
        elif ev["type"] == "content_block_delta" and ev["delta"]["type"] == "text_delta":
            text += ev["delta"]["text"]
            on_text(ev["delta"]["text"])
        elif ev["type"] == "content_block_delta" and ev["delta"]["type"] == "input_json_delta":
            calls[ev["index"]]["json"] += ev["delta"]["partial_json"]
            if on_args:
                on_args(calls[ev["index"]])
        elif ev["type"] == "content_block_stop" and ev["index"] in calls:
            calls[ev["index"]]["closed"] = True
        elif ev["type"] == "message_delta":
            stop = ev["delta"]["stop_reason"]
    return {"text": text, "calls": list(calls.values()), "stop": stop}

turn = assemble(events)
print("\n", turn)

Saving it no

w.


 {'text': 'Saving it now.', 'calls': [{'id': 'toolu_11', 'name': 'write_file', 'json': '{"filename": "mars_day.txt", "text": "A day on Mars lasts about 1479 minutes."}', 'closed': True}], 'stop': 'tool_use'}


The text printed live. The buffer holds the whole argument string, the block is closed and the stop
reason is `tool_use`. Now, and only now, `json.loads` is safe.

In [8]:
args = json.loads(turn["calls"][0]["json"])
print(type(args).__name__, args)

dict {'filename': 'mars_day.txt', 'text': 'A day on Mars lasts about 1479 minutes.'}


### 5. A live preview of the arguments
A UI may want to show "Saving to mars_day.txt..." while the text is still arriving. A **partial JSON
parser** turns an unfinished string into the best object so far. `jiter` (Pydantic's JSON parser,
installed with the SDKs) does this with `partial_mode`. Use it for display only.

In [9]:
import jiter

def preview(call: dict) -> None:
    so_far = jiter.from_json(call["json"].encode(), partial_mode="trailing-strings")
    print(f"  {call['name']} {so_far}")

turn = assemble(fake_events(SAVE), on_text=lambda t: None, on_args=preview)

  write_file {}
  write_file {'filename': 'mars_day.t'}
  write_file {'filename': 'mars_day.txt'}
  write_file {'filename': 'mars_day.txt', 'text': 'A day on M'}
  write_file {'filename': 'mars_day.txt', 'text': 'A day on Mars lasts ab'}
  write_file {'filename': 'mars_day.txt', 'text': 'A day on Mars lasts about 1479 min'}
  write_file {'filename': 'mars_day.txt', 'text': 'A day on Mars lasts about 1479 minutes.'}


The preview grows piece by piece, and `filename` starts to appear after the second delta and is complete after the third. With
`partial_mode="trailing-strings"` an unfinished string is kept (`'mars_day.t'`); with `partial_mode=True`
it is dropped. Either way the object can be missing keys or hold half a value.

### 6. A tool call cut by `max_tokens`
Lesson 15 named this failure: truncated tool JSON. Give the same reply a budget of 6 deltas (2 for the sentence, 4 for the arguments).

In [10]:
cut = assemble(fake_events(SAVE, max_tokens=6), on_text=lambda t: None)
print("stop:", cut["stop"])
print("buffer:", cut["calls"][0]["json"])
print("partial parse:", jiter.from_json(cut["calls"][0]["json"].encode(), partial_mode="trailing-strings"))

stop: max_tokens
buffer: {"filename": "mars_day.txt", "text": "A day on M
partial parse: {'filename': 'mars_day.txt', 'text': 'A day on M'}


The signal to check is the stop reason, not the block state: Anthropic's docs do not say whether a
stop event arrives for a cut block. `max_tokens` with a tool call means the call is incomplete
(Anthropic's stop-reason guide: retry with a higher `max_tokens`).

The partial parse looks valid: `{'filename': 'mars_day.txt', 'text': 'A day on M'}`. Running
`write_file` with it would save half a sentence and report success. The rule: if the stop reason
is `max_tokens` and the reply has a tool call, throw the turn away and ask again with a bigger budget.

### 7. From events to a loop reply
`to_reply` turns an assembled turn into lesson 35's assistant message, or marks it `truncated`.

In [11]:
def to_reply(turn: dict) -> dict:
    if turn["stop"] == "max_tokens" and turn["calls"]:
        return {"role": "assistant", "truncated": True}
    calls = [{"id": c["id"], "name": c["name"], "args": json.loads(c["json"] or "{}")}
             for c in turn["calls"]]
    if calls:
        return {"role": "assistant", "text": turn["text"], "tool_calls": calls}
    return {"role": "assistant", "text": turn["text"]}

print(to_reply(assemble(fake_events(SAVE), on_text=lambda t: None)))
print(to_reply(cut))

{'role': 'assistant', 'text': 'Saving it now.', 'tool_calls': [{'id': 'toolu_11', 'name': 'write_file', 'args': {'filename': 'mars_day.txt', 'text': 'A day on Mars lasts about 1479 minutes.'}}]}
{'role': 'assistant', 'truncated': True}


A streaming fake model plays a script like lesson 35's `ScriptedModel`. It moves to the next reply
only when a turn was not truncated, so a retry gets the same reply with a bigger budget.

In [12]:
class FakeStreamingModel:
    def __init__(self, script: list, piece: int = 12):
        self.script, self.piece, self.turn = script, piece, 0

    def __call__(self, messages: list[dict], schemas: list[dict], max_tokens: int) -> dict:
        reply = self.script[min(self.turn, len(self.script) - 1)]
        reply = reply(messages) if callable(reply) else reply
        events = fake_events(reply, self.piece, max_tokens, turn=self.turn + 1)
        result = to_reply(assemble(events))
        if not result.get("truncated"):
            self.turn += 1
        return result

reply = FakeStreamingModel([SAVE])([], SCHEMAS, max_tokens=6)
print("\n", reply)

Saving it no

w.


 {'role': 'assistant', 'truncated': True}


### 8. The streaming loop
Lesson 35's `run_agent` with two changes. The model prints text as it streams. A truncated turn is
not added to the history: the loop retries it once with four times the budget, and gives up with
`stop="truncated"` if that fails too.

In [13]:
def run_agent_streaming(question, model, tools, schemas, max_steps=8, max_tokens=400) -> dict:
    messages, errors = [{"role": "user", "text": question}], 0
    for step in range(1, max_steps + 1):
        reply = model(messages, schemas, max_tokens)
        if reply.get("truncated"):
            print(f"\n[step {step}: tool call cut at {max_tokens} tokens, retrying with {max_tokens * 4}]")
            reply = model(messages, schemas, max_tokens * 4)
            if reply.get("truncated"):
                return {"answer": None, "stop": "truncated", "steps": step, "errors": errors, "messages": messages}
        messages.append(reply)
        if not reply.get("tool_calls"):
            print(f"\n[step {step}: answer]")
            return {"answer": reply["text"], "stop": "end_of_turn", "steps": step, "errors": errors, "messages": messages}
        for call in reply["tool_calls"]:
            result = run_tool(call, tools)
            errors += result["is_error"]
            print(f"\n[step {step}: {call['name']}({call['args']}) -> {result['content'][:60]}]")
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"], **result})
    return {"answer": None, "stop": "step_limit", "steps": max_steps, "errors": errors, "messages": messages}

Lesson 35's Mars question with a script. With a budget of 6 deltas, the search and the sum fit, but
the `write_file` call with a long text does not.

In [14]:
QUESTION = "How many minutes long is a day on Mars? Save the answer to mars_day.txt."
def last_result(messages): return json.loads(messages[-1]["content"])

SCRIPT = [
    {"text": "Checking your notes. ", "tool_calls": [{"name": "search_notes", "args": {"query": "day on Mars"}}]},
    {"tool_calls": [{"name": "calculator", "args": {"expression": "24*60+39"}}]},
    lambda m: {"tool_calls": [{"name": "write_file", "args": {"filename": "mars_day.txt", "text":
        f"A day on Mars lasts about {last_result(m):g} minutes (24 hours and 39 minutes)."}}]},
    {"text": "A day on Mars lasts about 1479 minutes. I saved that to mars_day.txt."},
]
run = run_agent_streaming(QUESTION, FakeStreamingModel(SCRIPT), TOOLS, SCHEMAS, max_tokens=6)
print(run["stop"], "| steps:", run["steps"], "| file:", (SCOUT_DIR / "mars_day.txt").read_text())

Checking you

r notes. 


[step 1: search_notes({'query': 'day on Mars'}) -> [{"id": 3, "text": "A day on Mars lasts about 24 hours and 3]

[step 2: calculator({'expression': '24*60+39'}) -> 1479]

[step 3: tool call cut at 6 tokens, retrying with 24]

[step 3: write_file({'filename': 'mars_day.txt', 'text': 'A day on Mars lasts about 1479 minutes (24 hours and 39 minutes).'}) -> "saved 65 characters to mars_day.txt"]
A day on Mar

s lasts abou

t 1479 minut

es. I saved 

that to mars

_day.txt.


[step 4: answer]
end_of_turn | steps: 4 | file: A day on Mars lasts about 1479 minutes (24 hours and 39 minutes).


Text appeared as it streamed, each tool ran only after its arguments were complete, and the cut
`write_file` call never ran: the retry with a budget of 24 deltas saved the full sentence.

### 9. Gemini's version of the same events
Gemini sends a whole function call in one chunk by default (RESEARCH.md: observed while building
this lesson). On Vertex AI, `FunctionCallingConfig(stream_function_call_arguments=True)` streams
them. Let's look at the raw parts of one streamed `write_file` call.

In [15]:
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)
STREAM_ARGS = types.ToolConfig(function_calling_config=types.FunctionCallingConfig(stream_function_call_arguments=True))

def declarations(schemas):
    return [types.Tool(function_declarations=[types.FunctionDeclaration(
        name=s["name"], description=s["description"], parameters_json_schema=s["parameters"]) for s in schemas])]

print(MODEL, "| streamed arguments:", STREAM_ARGS.function_calling_config.stream_function_call_arguments)

gemini-3.5-flash-lite | streamed arguments: True


Ask for a short file and print every part of every chunk (the thought signature is hidden: it is a
long opaque value).

In [16]:
cfg = types.GenerateContentConfig(max_output_tokens=400, tools=declarations(SCHEMAS),
                                  tool_config=STREAM_ARGS, automatic_function_calling=NO_AFC)
ask = "Save a 40-word explanation of a day on Mars (24 hours 39 minutes) to mars_intro.txt."
raw_chunks = list(client.models.generate_content_stream(model=MODEL, contents=ask, config=cfg))
for i, ch in enumerate(raw_chunks):
    for part in ch.candidates[0].content.parts or []:
        print(i, part.model_dump(exclude_none=True, exclude={"thought_signature"}))
print("finish:", raw_chunks[-1].candidates[0].finish_reason)

0 {'function_call': {'id': 'Pl47FRLP', 'name': 'write_file', 'will_continue': True}}
1 {'function_call': {'partial_args': [{'json_path': '$.filename', 'string_value': 'mars_intro.txt', 'will_continue': True}], 'will_continue': True}}
2 {'function_call': {'partial_args': [{'json_path': '$.filename', 'string_value': ''}], 'will_continue': True}}
3 {'function_call': {'partial_args': [{'bool_value': True, 'json_path': '$.overwrite'}], 'will_continue': True}}
4 {'function_call': {'partial_args': [{'json_path': '$.text', 'string_value': 'A day on Mars, known as a sol, is remarkably similar in length to a', 'will_continue': True}], 'will_continue': True}}
5 {'function_call': {'partial_args': [{'json_path': '$.text', 'string_value': ' day on Earth. It lasts approximately 24 hours and 39 minutes, giving Martian missions and rovers a', 'will_continue': True}], 'will_continue': True}}
6 {'function_call': {'partial_args': [{'json_path': '$.text', 'string_value': ' familiar daily cycle to adapt to 

The event types differ but the shape is the same. The first part opens the call with `id` and
`name` (and carries the thought signature, lesson 22). Then `partial_args` arrive, each a value
tagged with a JSON path like `$.text`; string values come in pieces to be joined, and
`will_continue: True` means more is coming. Non-string values come whole: here `overwrite` arrived as
`bool_value: True`. A `function_call` part with nothing in it closes the call.

### 10. Assemble Gemini's stream
`add_partial` puts one piece into the arguments dict. Scout's tools only use top-level arguments, so
a nested path is refused rather than guessed.

In [17]:
def add_partial(args: dict, piece) -> None:
    key = piece.json_path.removeprefix("$.")
    if not key.isidentifier():
        raise ValueError(f"nested path {piece.json_path} is not handled here")
    if piece.string_value is not None:
        args[key] = args.get(key, "") + piece.string_value
    elif piece.number_value is not None:
        args[key] = piece.number_value
    elif piece.bool_value is not None:
        args[key] = piece.bool_value
    elif piece.null_value is not None:
        args[key] = None

args = {}
for value in ["A day ", "on Mars"]:
    add_partial(args, types.PartialArg(json_path="$.text", string_value=value, will_continue=True))
add_partial(args, types.PartialArg(json_path="$.overwrite", bool_value=True))
print(args)

{'text': 'A day on Mars', 'overwrite': True}


`gemini_assemble` reads the chunks: text is printed at once, a part with a `name` opens a call (or
holds a complete one when arguments were not streamed), `partial_args` fill it, an empty part closes
it. A finish reason of `MAX_TOKENS` with a call still open means truncated, as does
`MALFORMED_FUNCTION_CALL` (lesson 13).

In [18]:
def gemini_assemble(stream) -> dict:
    text, calls, finish = "", [], None
    for chunk in stream:
        cand = chunk.candidates[0]
        finish = cand.finish_reason or finish
        for part in (cand.content.parts if cand.content else None) or []:
            fc = part.function_call
            if part.text:
                text += part.text
                print(part.text, end="", flush=True)
            elif fc and fc.name:
                calls.append({"id": fc.id, "name": fc.name, "args": dict(fc.args or {}),
                              "sig": part.thought_signature, "done": fc.args is not None})
            elif fc and fc.partial_args:
                for piece in fc.partial_args:
                    add_partial(calls[-1]["args"], piece)
            elif fc:
                calls[-1]["done"] = True
    finish = finish.value if finish else None
    open_call = any(not c["done"] for c in calls)
    truncated = finish == "MALFORMED_FUNCTION_CALL" or (finish == "MAX_TOKENS" and open_call)
    return {"text": text, "calls": calls, "truncated": truncated, "finish": finish}

Run it on the chunks from step 9 (already received, so nothing new is sent).

In [19]:
g = gemini_assemble(raw_chunks)
print("finish:", g["finish"], "| truncated:", g["truncated"])
for c in g["calls"]:
    print(c["name"], c["done"], {k: v[:50] if isinstance(v, str) else v for k, v in c["args"].items()})

finish: STOP | truncated: False
write_file True {'filename': 'mars_intro.txt', 'overwrite': True, 'text': 'A day on Mars, known as a sol, is remarkably simil'}


### 11. Scout on Gemini, streaming
The model wrapper builds the Gemini history like lesson 35's `to_contents`. Our tool-call turns
are rebuilt from the assembled pieces, so each call's thought signature is put back on its part.

In [20]:
SYSTEM = ("You are Scout. Answer questions about the user's notes. Search the notes first. "
          "Use the calculator for arithmetic. Answer in one sentence.")

def to_contents(messages: list[dict]) -> list:
    contents = []
    for m in messages:
        if m["role"] == "user":
            contents.append(types.Content(role="user", parts=[types.Part(text=m["text"])]))
        elif m["role"] == "assistant":
            contents.append(m["raw"])
        else:
            key = "error" if m["is_error"] else "output"
            part = types.Part(function_response=types.FunctionResponse(
                id=m["call_id"], name=m["name"], response={key: json.loads(m["content"]) if not m["is_error"] else m["content"]}))
            if contents[-1].role == "user" and contents[-1].parts[0].function_response:
                contents[-1].parts.append(part)
            else:
                contents.append(types.Content(role="user", parts=[part]))
    return contents

The wrapper has the fake's call shape `(messages, schemas, max_tokens)`, so the streaming loop does
not change.

In [21]:
class GeminiStreamingModel:
    def __call__(self, messages: list[dict], schemas: list[dict], max_tokens: int) -> dict:
        cfg = types.GenerateContentConfig(system_instruction=SYSTEM, max_output_tokens=max_tokens,
                                          tools=declarations(schemas), tool_config=STREAM_ARGS,
                                          automatic_function_calling=NO_AFC)
        stream = client.models.generate_content_stream(model=MODEL, contents=to_contents(messages), config=cfg)
        g = gemini_assemble(stream)
        if g["truncated"]:
            return {"role": "assistant", "truncated": True}
        parts = [types.Part(text=g["text"])] if g["text"] else []
        parts += [types.Part(function_call=types.FunctionCall(id=c["id"], name=c["name"], args=c["args"]),
                             thought_signature=c["sig"]) for c in g["calls"]]
        raw = types.Content(role="model", parts=parts)
        calls = [{"id": c["id"], "name": c["name"], "args": c["args"]} for c in g["calls"]]
        if calls:
            return {"role": "assistant", "text": g["text"], "tool_calls": calls, "raw": raw}
        return {"role": "assistant", "text": g["text"], "raw": raw}

Now the real thing. Scout must save an explanation of about 80 words, which does not fit in a first
budget of 60 output tokens, so expect a truncated call and one retry.

In [22]:
for old in SCOUT_DIR.glob("*.txt"):
    old.unlink()
question = ("How long is a day on Mars in minutes? Then write an explanation of about 80 words for a "
            "12-year-old and save it to mars_day.txt.")
run = run_agent_streaming(question, GeminiStreamingModel(), TOOLS, SCHEMAS, max_tokens=60)
print(run["stop"], "| steps:", run["steps"], "| errors:", run["errors"])
saved = SCOUT_DIR / "mars_day.txt"
print(saved.read_text() if saved.exists() else "no file saved")


[step 1: search_notes({'query': 'Mars'}) -> [{"id": 1, "text": "Mars has two small moons, Phobos and Dei]

[step 2: calculator({'expression': '24*60+39'}) -> 1479]

[step 3: tool call cut at 60 tokens, retrying with 240]



[step 3: write_file({'filename': 'mars_day.txt', 'overwrite': True, 'text': 'A day on Mars, often called a sol, is just a little bit longer than a day on Earth! While Earth takes 24 hours to spin around once, Mars takes 24 hours and 39 minutes, which adds up to 1,479 minutes in total. That means a Martian day is only about 39 minutes longer than our days, giving you just a tiny bit more time to play video games or hang out with friends if you ever lived on the Red Planet!'}) -> "saved 399 characters to mars_day.txt"]


A day

 on Mars is 1,479 minutes long, and I have saved an explanation of it

 for a 12-year-old to mars_day.txt.


[step 4: answer]
end_of_turn | steps: 4 | errors: 0
A day on Mars, often called a sol, is just a little bit longer than a day on Earth! While Earth takes 24 hours to spin around once, Mars takes 24 hours and 39 minutes, which adds up to 1,479 minutes in total. That means a Martian day is only about 39 minutes longer than our days, giving you just a tiny bit more time to play video games or hang out with friends if you ever lived on the Red Planet!


Read the trace above. The search and the sum fit in 60 tokens. The `write_file` call did not: the
loop printed a retry line, threw that turn away, and the second attempt with 240 tokens saved the
whole explanation. The last step streamed Scout's one-sentence answer.

## At work
- **Anthropic's tool arguments stream as `input_json_delta` pieces**; its docs say to accumulate them
  and parse after `content_block_stop`, using a partial parser only for display. Teams put this in one
  adapter per vendor, because Gemini's path-and-value pieces look nothing like it. → lesson 37
- **`max_tokens` with an open tool call is an incomplete call**: Anthropic's stop-reason guide says
  retry with a higher `max_tokens`. Thinking tokens count against the same budget, which is how a
  budget that "always worked" starts cutting calls. → lesson 40
- **A stream that breaks mid-tool-call cannot be resumed**: Anthropic's docs note tool-use blocks
  cannot be partially recovered, so the whole turn is retried. If the tool has side effects, the
  retry must not repeat them. → lesson 39
- **Production logs record the stop reason and truncation retries per step**, so a rise in cut
  tool calls shows up before users notice. → lesson 49

## What just happened
- A streamed reply is a list of typed events: block start, deltas, block stop, stop reason.
- Text deltas were printed at once; tool-argument deltas were buffered by block index.
- Arguments were parsed only after the stop reason was known; a partial parser gave a live preview.
- A call cut by `max_tokens` was never run: the loop dropped the turn and retried with 4× the budget.
- Gemini on Vertex streamed the same information as `partial_args` (JSON path + value), and the same
  loop ran Scout on it with a truncation and a successful retry.

## Common mistakes
**1. Parsing each piece as it arrives.** A piece of JSON is not JSON.

In [23]:
for ev in fake_events(SAVE):
    if ev["type"] == "content_block_delta" and ev["delta"]["type"] == "input_json_delta":
        print(json.loads(ev["delta"]["partial_json"]))

JSONDecodeError: Expecting value: line 1 column 13 (char 12)

Fix: append to a buffer and call `json.loads` once, after the stop reason says the call is complete
(step 4).

**2. Keying buffers by the call id.** The id is only in `content_block_start`; deltas carry the
block `index`.

In [24]:
buffers = {}
for ev in fake_events(SAVE):
    if ev["type"] == "content_block_delta" and ev["delta"]["type"] == "input_json_delta":
        buffers[ev["id"]] = buffers.get(ev["id"], "") + ev["delta"]["partial_json"]

KeyError: 'id'

Fix: key by `ev["index"]`, and keep the id and name you saved when the block started.

In [25]:
buffers = {}
for ev in fake_events(SAVE):
    if ev["type"] == "content_block_delta" and ev["delta"]["type"] == "input_json_delta":
        buffers[ev["index"]] = buffers.get(ev["index"], "") + ev["delta"]["partial_json"]
print({i: json.loads(b) for i, b in buffers.items()})

{1: {'filename': 'mars_day.txt', 'text': 'A day on Mars lasts about 1479 minutes.'}}


**3. Running the partial parse of a cut call.** No error is raised, which is the danger: the tool
runs with half an argument and reports success.

In [26]:
half = jiter.from_json(cut["calls"][0]["json"].encode(), partial_mode="trailing-strings")
print(write_file(**half, overwrite=True), "->", repr((SCOUT_DIR / half["filename"]).read_text()))

saved 10 characters to mars_day.txt -> 'A day on M'


Fix: check the stop reason first and use `to_reply`, which refuses the turn.

In [27]:
print(to_reply(cut))

{'role': 'assistant', 'truncated': True}


## Try it
A UI should say "Saving to mars_day.txt..." as soon as the **file name is final**, not while it is
still arriving. Write `final_fields(buffer)` that returns only the arguments whose values are
complete (hint: compare `partial_mode=True` with `"trailing-strings"`). Then stream `SAVE` with
`piece=4` and print the number of the argument delta (counting only `input_json_delta` events,
starting at 1) at which `filename` first appears in `final_fields`.

In [28]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [29]:
llm_replay.stop()

## Key terms
- **stream event** — one typed message in a streamed reply, such as Anthropic's `content_block_start`, `content_block_delta`, `content_block_stop` and `message_delta`.
- **input_json_delta** — Anthropic's delta type for tool arguments: a `partial_json` string to append to the call's buffer.
- **partial JSON** — an unfinished JSON string; not valid until the last piece arrives, so parse only after the call is complete.
- **partial JSON parser** — a parser that turns unfinished JSON into the best object so far (`jiter.from_json(..., partial_mode=...)`); for previews, never for running tools.
- **partial_args** — Gemini's (Vertex AI) streamed tool arguments: values tagged with a JSON path like `$.text`, with `will_continue` while more is coming.
- **truncated tool JSON** (see lessons 13 and 15) — the streaming continuation of that idea: the stream ends mid-arguments because `max_tokens` ran out, so the assembled JSON is incomplete; drop the turn and retry with a larger budget.